# Train a world model directly from Hugging Face Buckets

[`stable-worldmodel`](https://github.com/galilai-group/stable-worldmodel) is an open platform for **reproducible world-model research** — collecting data, training, and evaluating with model-predictive control across a large suite of standardized environments ([paper](https://arxiv.org/abs/2605.21800)).

This notebook demonstrates the piece that makes large-scale world-model training practical on throwaway hardware: **streaming a multi-million-frame dataset straight from Hugging Face Buckets, with no local download.**

World-model datasets are large — millions of frames of pixels, actions, and states. Copying them onto every training box is slow and wasteful, and a non-starter on short-lived spot / Colab GPUs. `stable-worldmodel` stores datasets in the [**Lance**](https://lancedb.github.io/lance/) columnar format and reads them over `hf://buckets/...` using HTTP **ranged reads**, so the trainer fetches only the rows each batch needs — directly from object storage, with no sync step.

In a few minutes on a free Colab GPU you'll:

1. **Authenticate** with Hugging Face
2. **Install** `stable-worldmodel` and the Lance backend
3. **Open** a ~2M-frame PushT dataset directly from HF Buckets (no download)
4. **Train** one epoch of [**LeWorldModel**](https://le-wm.github.io/), reading batches live from the bucket

> ⚙️ **Before you start:** enable a GPU runtime via **Runtime → Change runtime type → GPU** (a free T4 is enough).


## 1. Hugging Face auth



In [ ]:
import os
from getpass import getpass

# A Hugging Face *account* token is required (anonymous IPs get rate-limited).
# In Colab, store it once under Secrets (the key icon in the left sidebar) as
# `HF_TOKEN`; otherwise you'll be prompted to paste it. The token is not saved
# in the notebook.
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    os.environ['HF_TOKEN'] = getpass('Hugging Face token: ')


## 2. Install dependencies


In [ ]:
!pip install --pre --quiet \
    --extra-index-url https://pypi.fury.io/lance-format/ \
    --extra-index-url https://pypi.fury.io/lancedb/ \
    'pylance' 'lancedb' \
    stable-worldmodel \
    stable-pretraining \
    opencv-python-headless imageio imageio-ffmpeg huggingface_hub torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 MB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 MB 12.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.2/305.2 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.8/334.8 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.1/56.1 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 72.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.2/154.2 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.3/233.3 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!git clone https://github.com/galilai-group/stable-worldmodel

Cloning into 'stable-worldmodel'...
remote: Enumerating objects: 5818, done.
remote: Counting objects: 100% (2026/2026), done.
remote: Compressing objects: 100% (740/740), done.
remote: Total 5818 (delta 1433), reused 1286 (delta 1286), pack-reused 3792 (from 3)
Receiving objects: 100% (5818/5818), 401.14 MiB | 33.58 MiB/s, done.
Resolving deltas: 100% (3671/3671), done.


## 3. Open the dataset directly from HF Buckets

Same code path the trainer uses. Measures cold first-batch latency and confirms row + episode counts.

In [ ]:
import time
from stable_worldmodel.data.formats.lance import Lance

URI = 'hf://buckets/galilai-group/swm/pusht_expert_train.lance'

t = time.perf_counter()
dataset = Lance.open_reader(
    URI,
    num_steps=4,
    frameskip=5,
    keys_to_load=['pixels', 'action', 'proprio'],
)
print(f'open_reader: {time.perf_counter() - t:.2f}s')
print(f'  windows : {len(dataset):,}')
print(f'  episodes: {len(dataset.lengths):,}')
print(f'  columns : {dataset.column_names}')

open_reader: 2.80s
  windows : 1,981,721
  episodes: 18,685
  columns : ['pixels', 'action', 'proprio']


## 4. one-epoch LeWM training run

Train directly from object storage using the Lance dataset — no cluster setup and no large dataset download step needed.

In [ ]:
#%cd stable-worldmodel
!python scripts/train/lewm.py data.dataset.name='hf://buckets/galilai-group/swm/pusht_expert_train.lance' loader.batch_size=32 loader.num_workers=2

07:03:42 | INFO  | __init__.py | TensorFlow version 2.20.0 available.
07:03:42 | INFO  | __init__.py | JAX version 0.7.2 available.
07:03:47 | INFO  | atomic_chec~| [atomic_save] installed crash-safe checkpoint plugin (write to sibling .tmp + fsync + atomic rename)
Loading dataset "hf://buckets/galilai-group/swm/pusht_expert_train.lance" from default location
[2026-06-02 07:03:50,711][root][WARNING] - LanceDataset: keys_to_cache=['action', 'proprio', 'state'] is not required — Lance has efficient random access via batched __getitems__.
07:04:22 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
07:04:23 | INFO  | module.py   | Setting up DataModule
07:04:23 | INFO  | utils.py    | ── Module ────────────────────────────────────────
07:04:23 | WARN  | module.py   | ! No hyperparameters provided - hyperparameter logging is disabled.
07:04:2